In [3]:
#import required libs
import numpy as np
import pandas as pd
import os
from dotenv import load_dotenv
import json
from mistralai import Mistral

In [4]:
# Load Json Dataset from .json

with open('documents.json', 'rt', encoding='utf-8') as f_out:
    raw_docs = json.load(f_out)

In [8]:
documents = []
for course_dict in raw_docs:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)

In [12]:
from elasticsearch import Elasticsearch

In [14]:
es_client = Elasticsearch('http://localhost:9200')

In [28]:
query = "How can I run Kafka?"

In [29]:
search_query = {
    "size": 5,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^3", "text", "section"],
                    "type": "best_fields"
                }
            },
            "filter": {
                "term": {
                    "course": "data-engineering-zoomcamp"
                }
            }
        }
    }
}

In [20]:
index_name = 'zoom-camp'

In [24]:
es_client.indices.create(index=index_name, body=index_settings)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'zoom-camp'})

In [21]:
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

In [31]:
from tqdm.auto import tqdm

In [33]:
for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

  0%|          | 0/948 [00:00<?, ?it/s]

In [34]:
results = es_client.search(index=index_name, body=search_query)

In [37]:
search_results = []

for doc in results['hits']['hits']:
    search_results.append(doc['_source'])

In [43]:
def elastic_search(query):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": "data-engineering-zoomcamp"
                    }
                }
            }
        }
    }
    answer = []
    results = es_client.search(index=index_name, body=search_query)
    for doc in results['hits']['hits']:
        answer.append(doc['_source'])
    return answer

In [44]:
results_2 = elastic_search(query)

In [54]:
def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant.\n
ANSWER the QUESTION based on the CONTEXT from the FAQ database. \n
Use only the facts from the CONTEXT when answering the QUESTION.\n
If the user's question doesn't contain in the FAQ database, please just kindly decline the requrest and reponse in a kind manner.\n
Don't add any other extra words.\n
QUESTION: {question}

CONTEXT:
{context}
""".strip()
    
    context = ""
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"
    prompt = prompt_template.format(question=query, context=context)

    return prompt

In [55]:
prompt = build_prompt(query, results_2)

In [56]:
print(prompt)

You're a course teaching assistant.

ANSWER the QUESTION based on the CONTEXT from the FAQ database. 

Use only the facts from the CONTEXT when answering the QUESTION.

If the user's question doesn't contain in the FAQ database, please just kindly decline the requrest and reponse in a kind manner.

Don't add any other extra words.

QUESTION: How can I run Kafka?

CONTEXT:
section: Module 6: streaming with kafka
question: Confluent Kafka: Where can I find schema registry URL?
answer: In Confluent Cloud:
Environment → default (or whatever you named your environment as) → The right navigation bar →  “Stream Governance API” →  The URL under “Endpoint”
And create credentials from Credentials section below it

section: Module 6: streaming with kafka
question: Java Kafka: How to run producer/consumer/kstreams/etc in terminal
answer: In the project directory, run:
java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java

section: Workshop 1 - dlthub
quest

In [68]:
# Connecting LLMs APIs
load_dotenv()

api_key = os.getenv('MISTRAL_API_KEY_2')
client = Mistral(api_key=api_key)

In [70]:
# Buidling chat model
model = "mistral-small-2506"

def llm(prompt):
    try:
        response = client.chat.complete(
            model = model,
            messages = [{
                'role': 'user',
                'content': prompt
            }]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"An error occured: {e}")

In [71]:
answer = llm(prompt)

In [72]:
print(answer)

In the project directory, run:
java -cp build/libs/<jar_name>-1.0-SNAPSHOT.jar:out src/main/java/org/example/JsonProducer.java


In [76]:
query = "The course started, Can I still join?"


In [80]:
# Building Rag_bot:
def rag_bot():
    print("--- The Program has started ---")
    print("--- Welcome to the ChatBot ---\n")
    while True:
        query = input("You: ")
        if query != 'exit' and query != 'quit':
            search_res = elastic_search(query)
            prompt = build_prompt(query, search_res)
            answer = llm(prompt)
            print(f"Bot: {answer}\n")
        else:
            print("Bot: GoodBye!")
            print("\n--- End of the Program ---")
            break

In [112]:
rag_bot()

--- The Program has started ---
--- Welcome to the ChatBot ---



You:  How do copy a file to a Docker container


Bot: To copy a file to a Docker container, you can use the `docker cp` command. For example:

```bash
docker cp /path/to/local/file container_name:/path/in/container
```

This command copies the specified local file to the specified path inside the container.



You:  exit


Bot: GoodBye!

--- End of the Program ---


In [84]:
# --- End of the program ---

In [85]:
#  --- Start of the Homework ---

In [125]:
search_query = {
    "size": 5,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^3", "text"],
                    "type": "best_fields"
                }
            },
            "filter": {
                "term": {
                    "course": "data-engineering-zoomcamp"
                }
            }
        }
    }
}

In [88]:
query = "How do execute a command on a Kubernates pod?"

In [91]:
search_res = es_client.search(index=index_name, body=search_query)

In [92]:
search_res

ObjectApiResponse({'took': 8, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 334, 'relation': 'eq'}, 'max_score': 31.973522, 'hits': [{'_index': 'zoom-camp', '_id': 'z3EvXpgBU6B9DdNo9A70', '_score': 31.973522, '_source': {'text': 'Install the astronomer-cosmos package as a dependency. (see Terraform example).\nMake a new folder, dbt/, inside the dags/ folder of your Composer GCP bucket and copy paste your dbt-core project there. (see example)\nEnsure your profiles.yml is configured to authenticate with a service account key. (see BigQuery example)\nCreate a new DAG using the DbtTaskGroup class and a ProfileConfig specifying a profiles_yml_filepath that points to the location of your JSON key file. (see example)\nYour dbt lineage graph should now appear as tasks inside a task group like this:', 'section': 'Course Management Form for Homeworks', 'question': 'How to run a dbt-core project as an Airflow Task Group on Goo

In [94]:
answer_3 = results['hits']['hits'][0]['_score']

In [96]:
print(f"The answer is : {answer_3}")

The answer is : 29.75321


In [114]:
query = "How do copy a file to a Docker container?"

In [132]:
search_query = {
    "size": 3,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^4", "text"],
                    "type": "best_fields"
                }
            },
            "filter": {
                "term": {
                    "course": "machine-learning-zoomcamp"
                }
            }
        }
    }
}

In [133]:
search_res = es_client.search(index=index_name, body=search_query)

In [161]:
context = ""
for doc in search_res['hits']['hits']:
    context = context + f"Q: {doc['_source']['question']}\nA: {doc['_source']['text']}\n\n"

In [162]:
print(context)

Q: How do I debug a docker container?
A: Launch the container image in interactive mode and overriding the entrypoint, so that it starts a bash command.
docker run -it --entrypoint bash <image>
If the container is already running, execute a command in the specific container:
docker ps (find the container-id)
docker exec -it <container-id> bash
(Marcos MJD)

Q: How do I copy files from my local machine to docker container?
A: You can copy files from your local machine into a Docker container using the docker cp command. Here's how to do it:
To copy a file or directory from your local machine into a running Docker container, you can use the `docker cp command`. The basic syntax is as follows:
docker cp /path/to/local/file_or_directory container_id:/path/in/container
Hrithik Kumar Advani

Q: How do I copy files from a different folder into docker container’s working directory?
A: You can copy files from your local machine into a Docker container using the docker cp command. Here's how to 

In [163]:
# Question - 5

prompt_template = """
You're a course teaching assistant.\n
Answer the QUESTION based on the CONTEXT from the FAQ database.\n
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

In [164]:
prompt = prompt_template.format(question=query, context=context)

In [166]:
print(prompt)

You're a course teaching assistant.

Answer the QUESTION based on the CONTEXT from the FAQ database.

Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: How do copy a file to a Docker container?

CONTEXT:
Q: How do I debug a docker container?
A: Launch the container image in interactive mode and overriding the entrypoint, so that it starts a bash command.
docker run -it --entrypoint bash <image>
If the container is already running, execute a command in the specific container:
docker ps (find the container-id)
docker exec -it <container-id> bash
(Marcos MJD)

Q: How do I copy files from my local machine to docker container?
A: You can copy files from your local machine into a Docker container using the docker cp command. Here's how to do it:
To copy a file or directory from your local machine into a running Docker container, you can use the `docker cp command`. The basic syntax is as follows:
docker cp /path/to/local/file_or_directory container_id:/path/in/conta

In [167]:
print(len(prompt))

1450


In [168]:
!pip install tiktoken


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [172]:
answer = llm(prompt)

In [173]:
print(answer)

To copy a file to a Docker container, you can use the `docker cp` command. The syntax is:

```
docker cp /path/to/local/file_or_directory container_id:/path/in/container
```

For example, to copy a file named `example.txt` from your local machine to a running container with ID `abc123`, you would use:

```
docker cp example.txt abc123:/path/in/container
```


In [179]:
response.usage.total_tokens

571